# Python's PyTest Generator and Runner

This simple project captures the learnings from week 4.
It takes in a python function and generates testing code using huggingface's `InferenceClient` and `Qwen` as the model.

The user has an opportunity to run and verify the testing code.

- **Dynamic Execution**: Runs pytest on code strings via tempfile for seamless discovery.
- **Output Capture**: Intercepts sys.stdout into io.StringIO to fetch terminal logs.
- **Forced Color**: Uses PY_COLORS=1 and --color=yes to generate ANSI codes.
- **ANSI to HTML**: Converts terminal codes to inline-styled HTML using ansi2html.
- **Gradio UI**: Displays results in a gr.HTML container for a "real" console feel.


In [ ]:
!pip install pytest ansi2html

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, InferenceClient
import gradio as gr

In [ ]:
load_dotenv()

In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct"
client = InferenceClient(MODEL_ID, token=HF_TOKEN)

SYSTEM_PROMPT = """
You are an Senior QA Engineer.
Generate pytest unit tests for the given code.
Include:
- Happy path tests
- Edge cases
- Error handling tests
- Mock any side effects
Keep tests simple and clear.
"""


def generate_tests(code):
    user_prompt = f"""
Generate unit tests for this code:\n\n{code}
IMPORTANT: Do not include the import for the function being tested.
IMPORTANT: Do not include any explanation.
IMPORTANT: Only return the code
"""
    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            stream=True
        )

        result = ""
        for chunk in response:
            if chunk.choices[0].delta.content:
                result += chunk.choices[0].delta.content
                yield result.replace("```python", "").replace("```", "")

    except Exception as e:
        yield f"Error: {str(e)}"


In [ ]:
import pytest
import io
import sys
import tempfile
from ansi2html import Ansi2HTMLConverter

def run_tests(user_code, test_code):
    os.environ["PY_COLORS"] = "1"
    os.environ["FORCE_COLOR"] = "1"

    # 1. Capture stdout for terminal output
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    # 2. Create a temporary file that pytest can "see"
    # The suffix '.py' is critical for pytest discovery
    with tempfile.NamedTemporaryFile(suffix=".py", mode="w", delete=False) as tmp:
        all_code = f"{user_code}\n\n{test_code}"
        tmp.write(all_code)
        tmp_path = tmp.name

    try:
        # 3. Run pytest on the temporary file
        # -q: quiet, --no-header: clean output, -W ignore: hide warnings
        pytest.main(["-q", "--no-header", "-W", "ignore", "--color=yes", tmp_path])
    except Exception as e:
        print(f"Error during execution: {e}")
    finally:
        # 4. Clean up: Restore stdout and delete the temp file
        sys.stdout = old_stdout
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

        # 5. Convert ANSI terminal codes to HTML for Gradio
        conv = Ansi2HTMLConverter(dark_bg=True, inline=True)
        html_output = conv.convert(buffer.getvalue(), full=False)
        # Wrap in a styled div to look like a terminal
        return f"""
        <div style="background-color: #0c0c0c; color: #cccccc; padding: 15px;
                    border-radius: 8px; font-family: monospace;
                    white-space: pre-wrap; border: 1px solid #333;">
            {html_output}
        </div>
        """

In [ ]:
gr.close_all()

code_example = """
def divide_numbers(numerator, denominator):
    \"""Divides two numbers and handles potential edge cases.""\"
    if not isinstance(numerator, (int, float)) or not isinstance(denominator, (int, float)):
        raise TypeError("Both inputs must be integers or floats.")

    if denominator == 0:
        raise ZeroDivisionError("Cannot divide by zero.")

    return numerator / denominator

"""

with gr.Blocks(title="Pytest Generator") as ui:
    gr.Markdown("## Paste your Python function and get AI-generated unit tests")
    gr.Markdown("## _Only include a single function_")

    with gr.Row():
        with gr.Column():
            code_input = gr.Code(
                label="Your Code",
                language="python",
                lines=20,
                max_lines=35,
                value=code_example
            )
            gr.Examples(
                example_labels=["code example: divide_numbers"],
                examples=[code_example],
                inputs=[code_input]
            )

            generate_btn = gr.Button("Generate Tests", variant="primary")

        with gr.Column(scale=1):
            tests_code_output = gr.Code(
                label="Generated Tests",
                language="python",
                lines=20,
                max_lines=35,
            )
            run_tests_btn = gr.Button("Run Tests", variant="huggingface")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## Test Results")
            tests_result = gr.HTML(label="Test Result")

    generate_btn.click(
        fn=generate_tests,
        inputs=[code_input],
        outputs=[tests_code_output]
    )

    run_tests_btn.click(
        fn=run_tests,
        inputs=[code_input, tests_code_output],
        outputs=[tests_result],

    )

ui.launch()